In [1]:
# ============================================================
# Centrality measures assesment and comparison (fast/guarded): distribution, stress-test, plausibility

# Input  : WRDS holdings-derived edges file
#          (WRDS/financed_emissions_network_final_plus_manual_fuzzy.csv

# Expect : (case-insensitive) columns ->
#            manager, stock, ownership (fraction or %), optional Scope1+2, market_value

# Output : In-memory DataFrames / printouts
#          - Centrality scores (per manager):
#              * in_degree, out_degree, weighted in/out
#              * betweenness (approximate)
#              * PageRank (α=0.85)
#              * HITS (authorities & hubs)
#              * eigenvector
#              * closeness, harmonic (only if graph is small enough)
#          - Distributional statistics:
#              * Gini coefficients
#              * Top-k shares
#              * Nonzero shares
#          - Stress-test results:
#              * AUC of largest component under targeted removals
#          - Plausibility checks:
#              * Correlation (Pearson / Spearman, raw and log1p) with AUM proxy

# Purpose: Compare influence metrics across managers in large equity networks,
#          assess their distributional behavior, robustness to removals,
#          and empirical plausibility versus market size.
# ============================================================
import re, time, math, numpy as np, pandas as pd, networkx as nx
from pathlib import Path
from collections import OrderedDict

T0 = time.perf_counter()
def tick(msg):
    print(f"[{time.perf_counter()-T0:7.2f}s] {msg}", flush=True)
    
# -----------------------------
# 1) Load & standardize columns
# -----------------------------
EDGES_BASE = Path("../../WRDS/financed_emissions_network_final_plus_manual.csv")
EDGES_FUZZ = Path("../../WRDS/financed_emissions_network_final_plus_manual_fuzzy.csv")
EDGES_FILE = EDGES_FUZZ if EDGES_FUZZ.exists() else EDGES_BASE
tick(f"Reading: {EDGES_FILE}")
df0 = pd.read_csv(EDGES_FILE, low_memory=False)
tick(f"Loaded {len(df0):,} rows; {df0.shape[1]} cols")

def norm_name(s: str) -> str:
    if not isinstance(s, str): s = "" if s is None else str(s)
    s = s.upper().replace("&", " AND ")
    s = re.sub(r"[^\w\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def find_first_col(df, candidates, also_contains=None, required=False, label=""):
    lc = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lc: 
            return lc[cand.lower()]
    if also_contains:
        for c in df.columns:
            cl = c.lower()
            if any(tok in cl for tok in also_contains):
                return c
    if required:
        raise KeyError(f"Missing required column {label or candidates} in {list(df.columns)}")
    return None

# keys
mgr_key = find_first_col(df0, ['_manager_key'])
cmp_key = find_first_col(df0, ['_company_key'])
if mgr_key is None:
    mgr_name_col = find_first_col(df0, ['manager','mgrname','(mgrname) manager name'], required=True, label='manager')
    df0['_manager_key'] = df0[mgr_name_col].map(norm_name)
else:
    df0['_manager_key'] = df0[mgr_key]
if cmp_key is None:
    stk_name_col = find_first_col(df0, ['stock','stkname','(stkname) stock name'], required=True, label='stock')
    df0['_company_key'] = df0[stk_name_col].map(norm_name)
else:
    df0['_company_key'] = df0[cmp_key]

# ownership fraction
own_frac_col = find_first_col(df0, ['own_frac','ownership','ownership_pct','pct_own','weight'],
                              also_contains=['own','pct','percent'], required=True, label='ownership')
df0['own_frac'] = pd.to_numeric(df0[own_frac_col], errors='coerce')
if np.nanmax(df0['own_frac'].to_numpy()) > 1.0:
    df0['own_frac'] = (df0['own_frac'] / 100.0).clip(lower=0.0)

# optional fields
s12_col = find_first_col(df0, ['s12','scope12','scope12_total_tco2','Scope 1+2','scope_1_2','scope1+2'],
                         also_contains=['scope1','scope 1','s1','s2'])
df0['s12'] = pd.to_numeric(df0[s12_col], errors='coerce').fillna(0.0) if s12_col else 0.0

mkt_candidates = ['market_cap','marketcap','market_value','mkt_cap','mv_usd','mv_eur',
                  'aum','assets_under_management','manager_aum','aum_usd']
mkt_col = find_first_col(df0, mkt_candidates, also_contains=['market','cap','aum'])
df0['market_value'] = pd.to_numeric(df0[mkt_col], errors='coerce') if mkt_col else np.nan

# tidy edges (issuer -> owner)
edges = (df0.loc[:, ['_company_key','_manager_key','own_frac','s12','market_value']]
            .dropna(subset=['_company_key','_manager_key','own_frac'])
            .query('own_frac > 0')
            .copy())
edges = edges[edges['_company_key'] != edges['_manager_key']]
row_sum = edges.groupby('_company_key')['own_frac'].transform('sum')
over = row_sum > 1.0
if over.any():
    edges.loc[over, 'own_frac'] = edges.loc[over, 'own_frac'] / row_sum[over]
tick(f"Edges after cleaning: {len(edges):,}")


[   0.00s] Reading: ../../WRDS/financed_emissions_network_final_plus_manual_fuzzy.csv
[   3.89s] Loaded 2,608,201 rows; 14 cols
[   4.93s] Edges after cleaning: 2,608,168


In [3]:
# -----------------------------------
# 2) Build a FAST issuer→owner graph (with backbone filtering)
# -----------------------------------
import time; t0=time.time()

# ---- knobs (tune these if still slow) ----
MAX_PER_ISSUER = 15          # keep top-K owners per issuer (strength by own_frac)
MIN_OWN_FRAC   = 0.0005      # drop edges below 0.05% ownership
MAX_EDGES_HARD = 600_000     # hard cap: keep globally top edges by own_frac if exceeded

print(f"[build] starting with {len(edges):,} edges")

# 2.1 — backbone: strongest edges per issuer
edges_sorted = edges.sort_values(['_company_key','own_frac'], ascending=[True, False])
edges_backbone = edges_sorted.groupby('_company_key', as_index=False).head(MAX_PER_ISSUER)

# 2.2 — threshold tiny ownerships
if MIN_OWN_FRAC is not None and MIN_OWN_FRAC > 0:
    edges_backbone = edges_backbone[edges_backbone['own_frac'] >= MIN_OWN_FRAC]

# 2.3 — hard global cap (optional)
if MAX_EDGES_HARD and len(edges_backbone) > MAX_EDGES_HARD:
    edges_backbone = edges_backbone.nlargest(MAX_EDGES_HARD, 'own_frac')

print(f"[build] after backbone & filters: {len(edges_backbone):,} edges")

# 2.4 — build graph in BULK (no per-row loop)
G = nx.DiGraph()

# add edges with weight 'w' in one shot
G.add_weighted_edges_from(
    zip(edges_backbone['_company_key'].to_numpy(),
        edges_backbone['_manager_key'].to_numpy(),
        edges_backbone['own_frac'].astype(float).to_numpy()),
    weight='w'
)

# add roles (vectorized-ish)
G.add_nodes_from(edges_backbone['_company_key'].unique(), role='issuer')
G.add_nodes_from(edges_backbone['_manager_key'].unique(), role='owner')

# add distance attribute for shortest-path based metrics (one bulk set)
dist_vals = 1.0 / np.maximum(edges_backbone['own_frac'].to_numpy(float), 1e-12)
edge_keys = list(zip(edges_backbone['_company_key'], edges_backbone['_manager_key']))
nx.set_edge_attributes(G, {e: {'dist': d} for e, d in zip(edge_keys, dist_vals)})

# 2.5 — keep only the giant weakly connected component (reduces work a lot)
UG = G.to_undirected()
if UG.number_of_nodes() > 0:
    largest = max(nx.connected_components(UG), key=len)
    G = G.subgraph(largest).copy()

n = G.number_of_nodes(); m = G.number_of_edges()
managers = [n_ for n_ in G.nodes() if G.in_degree(n_) > 0]
print(f"[build] giant component: {n:,} nodes, {m:,} edges | managers: {len(managers):,} | {time.time()-t0:,.1f}s")

# ---- speed knobs for later steps (you had these already; keep them here) ----
FAST = True
PR_ALPHA = 0.85
BETW_SAMPLES = max(32, min(128, int(0.01 * n))) if FAST else max(200, int(0.02 * n))
DO_CLOSENESS = n <= 12000 or not FAST
DO_HARMONIC  = n <= 12000 or not FAST


[build] starting with 2,608,168 edges
[build] after backbone & filters: 127,566 edges
[build] giant component: 16,103 nodes, 127,506 edges | managers: 5,717 | 4.2s


In [4]:
# -----------------------------------
# 3) Centralities (fast variants)
# -----------------------------------
centralities = OrderedDict()

tick("Centrality: degrees")
centralities['in_degree']     = dict(G.in_degree(weight=None))
centralities['out_degree']    = dict(G.out_degree(weight=None))
centralities['w_in_degree']   = dict(G.in_degree(weight='w'))
centralities['w_out_degree']  = dict(G.out_degree(weight='w'))

tick(f"Centrality: betweenness (approx, k={BETW_SAMPLES})")
centralities['betweenness'] = nx.betweenness_centrality(G, k=BETW_SAMPLES, normalized=True, weight='dist', seed=42)

tick("Centrality: PageRank")
centralities['pagerank'] = nx.pagerank(G, alpha=PR_ALPHA, weight='w', max_iter=200, tol=1e-08)

tick("Centrality: HITS (authorities/hubs)")
# use power iteration (faster, sparse-friendly)
hits_h, hits_a = nx.hits(G, max_iter=200, normalized=True)
centralities['hits_authority'] = hits_a
centralities['hits_hub']       = hits_h

tick("Centrality: eigenvector (power method)")
# power iteration (not numpy dense eig)
centralities['eigenvector'] = nx.eigenvector_centrality(G, max_iter=500, tol=1e-06, weight='w')

if DO_CLOSENESS:
    tick("Centrality: closeness (unweighted on UG for speed)")
    UG = G.to_undirected()  # fast proxy; set distance='dist' if you can afford it
    centralities['closeness'] = nx.closeness_centrality(UG, wf_improved=True)
else:
    tick("Skip closeness (graph too large)")

if DO_HARMONIC:
    tick("Centrality: harmonic (unweighted on UG for speed)")
    UG = UG if 'UG' in locals() else G.to_undirected()
    centralities['harmonic'] = nx.harmonic_centrality(UG)
else:
    tick("Skip harmonic (graph too large)")

[ 145.26s] Centrality: degrees
[ 145.35s] Centrality: betweenness (approx, k=128)
[ 146.65s] Centrality: PageRank
[ 147.25s] Centrality: HITS (authorities/hubs)
[ 147.43s] Centrality: eigenvector (power method)
[ 156.39s] Skip closeness (graph too large)
[ 156.39s] Skip harmonic (graph too large)


In [5]:

# -----------------------------------
# 4) Scores DF (managers only)
# -----------------------------------
def scores_df(centralities_dict, keep_nodes):
    df = pd.DataFrame({name: pd.Series(vals) for name, vals in centralities_dict.items()})
    df = df.loc[df.index.intersection(keep_nodes)].copy()
    mv = edges.groupby('_manager_key')['market_value'].max()
    df['market_value'] = mv.reindex(df.index).astype(float)
    return df.fillna(0.0)

C = scores_df(centralities, managers)
tick(f"Scored managers: {C.shape[0]:,}")



[ 166.28s] Scored managers: 5,717


In [6]:
# -----------------------------------
# 5) Distributional behavior
# -----------------------------------
def gini(x):
    x = np.asarray(x, float)
    x = x[x>=0]
    if x.size == 0: return np.nan
    if np.allclose(x.sum(), 0): return 0.0
    x_sorted = np.sort(x)
    n = x_sorted.size
    cumx = np.cumsum(x_sorted)
    g = 1.0 - 2.0 * np.sum(cumx) / (n * cumx[-1]) + 1.0/n
    return max(0.0, min(1.0, g))

def top_share(x, k):
    x = np.asarray(x, float)
    s = x.sum()
    if s <= 0: return 0.0
    return np.sort(x)[-k:].sum() / s

dist_rows = []
for col in [c for c in C.columns if c != 'market_value']:
    vals = C[col].values
    dist_rows.append({
        'metric': col,
        'gini': gini(vals),
        'top1_share': top_share(vals, 1),
        'top10_share': top_share(vals, min(10, len(vals))),
        'top50_share': top_share(vals, min(50, len(vals))),
        'nonzero_share': float(np.count_nonzero(vals))/max(1,len(vals))
    })
dist_table = pd.DataFrame(dist_rows).sort_values('gini', ascending=True)

tick("=== (a) Distributional behavior ===")
print(dist_table.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))



[ 177.39s] === (a) Distributional behavior ===
        metric   gini  top1_share  top10_share  top50_share  nonzero_share
      pagerank 0.5493      0.0469       0.1682       0.3080         1.0000
     in_degree 0.8352      0.0348       0.2221       0.4583         1.0000
   w_in_degree 0.8682      0.0720       0.2732       0.4537         1.0000
hits_authority 0.8955      0.0577       0.3345       0.5696         1.0000
    out_degree 0.9879      0.0145       0.1449       0.7246         0.0138
   eigenvector 0.9882      0.2772       0.7008       0.9532         1.0000
      hits_hub 0.9896      0.0190       0.1816       0.8272         0.0138
  w_out_degree 0.9906      0.0298       0.2372       0.8579         0.0138
   betweenness 0.9992      0.3237       0.9981       1.0000         0.0023


In [7]:
# -----------------------------------
# 6) Influence stress-test
# -----------------------------------
def fragmentation_curve(G, scores, ks=(1,5,10,25,50,100,250,500)):
    order = [n for n, _ in sorted(scores.items(), key=lambda t: t[1], reverse=True)]
    res = []
    UG = G.to_undirected()
    for k in ks:
        rem = set(order[:min(k, len(order))])
        H = UG.copy()
        H.remove_nodes_from(rem)
        if H.number_of_nodes() == 0:
            res.append({'k': k, 'largest_wcc_rel': 0.0, 'num_components': 0}); continue
        comps = list(nx.connected_components(H))
        largest = max((len(c) for c in comps), default=0)
        res.append({'k': k,
                    'largest_wcc_rel': largest / max(1, H.number_of_nodes()),
                    'num_components': len(comps)})
    return pd.DataFrame(res)

ks = (1,5,10,25,50,100,250,500)
frag_results = {}
for metric in [c for c in C.columns if c != 'market_value']:
    tick(f"Stress-test: {metric}")
    frag_results[metric] = fragmentation_curve(G, C[metric].to_dict(), ks=ks)

def auc_lcc(df):
    x = np.array(df['k'], float)
    y = np.array(df['largest_wcc_rel'], float)
    return np.trapz(y, x) / (x[-1] - x[0] + 1e-12)

frag_summary = pd.DataFrame({
    'metric': list(frag_results.keys()),
    'AUC_largestWCC': [auc_lcc(frag_results[m]) for m in frag_results]
}).sort_values('AUC_largestWCC', ascending=True)

tick("=== (b) Influence stress-test summary (smaller AUC = fragments faster) ===")
print(frag_summary.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))


[ 188.82s] Stress-test: in_degree
[ 192.36s] Stress-test: out_degree
[ 195.94s] Stress-test: w_in_degree
[ 199.44s] Stress-test: w_out_degree
[ 203.00s] Stress-test: betweenness
[ 206.59s] Stress-test: pagerank
[ 210.21s] Stress-test: hits_authority
[ 213.82s] Stress-test: hits_hub
[ 217.39s] Stress-test: eigenvector
[ 220.97s] === (b) Influence stress-test summary (smaller AUC = fragments faster) ===
        metric  AUC_largestWCC
      pagerank          0.9224
     in_degree          0.9352
hits_authority          0.9353
   eigenvector          0.9578
   w_in_degree          0.9589
   betweenness          0.9649
    out_degree          0.9716
  w_out_degree          0.9716
      hits_hub          0.9718


/var/folders/gw/vnsb934j7fvdc3rjq0d035h00000gn/T/ipykernel_1526/124126592.py:30: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(y, x) / (x[-1] - x[0] + 1e-12)


In [10]:
# ============================================
# (7) Empirical plausibility NaN -> Empirical plausibility vs market value (AUM proxy)
# Requires: centrality bake-off already run -> DataFrame `C` indexed by _manager_key
# ============================================
import numpy as np, pandas as pd, re
from pathlib import Path
from scipy.stats import spearmanr

assert 'C' in globals(), "Run the centrality bake-off first to create `C`."

# ---------- locate a WRDS holdings file ----------
def first_existing(paths):
    for p in paths:
        if Path(p).exists(): return str(p)
    raise FileNotFoundError("None of the WRDS files found:\n" + "\n".join(map(str, paths)))

WRDS_CANDIDATES = [
    "../../WRDS/final_deduplicated_holdings.csv",
    "../../WRDS/WRDS_complete.csv",
    "../WRDS/final_deduplicated_holdings.csv",
    "../WRDS/WRDS_complete.csv",
]
WRDS_FILE = first_existing(WRDS_CANDIDATES)
print(f"[AUM] Using WRDS file: {WRDS_FILE}")

dfw = pd.read_csv(WRDS_FILE, low_memory=False)

# ---------- helpers ----------
def norm_name(s: str) -> str:
    if not isinstance(s, str): s = "" if s is None else str(s)
    s = s.upper().replace("&", " AND ")
    s = re.sub(r"[^\w\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def find_first_col(df, candidates, also_contains=None):
    lc = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lc: return lc[cand.lower()]
    if also_contains:
        for c in df.columns:
            cl = c.lower()
            if any(tok in cl for tok in also_contains): return c
    return None

# ---------- detect columns ----------
mgr_col = find_first_col(dfw, ['_manager_key','manager','(mgrname) manager name','mgrname'])
val_col = find_first_col(dfw, ['value','market_value','holding_value','mv'])
sh_col  = find_first_col(dfw, ['shares','(shares) shares held at end of qtr','shares_held'])
pr_col  = find_first_col(dfw, ['price','prc','(prc) share price, as of fdate'])
dt_col  = find_first_col(dfw, ['report_date','(rdate) report date','rdate','date','period'])

if mgr_col is None:
    raise KeyError("Manager name/ID column not found in WRDS file.")
if val_col is None and (sh_col is None or pr_col is None):
    raise KeyError("Need either a holdings 'value' column or both 'shares' and 'price' columns.")

# ---------- normalize keys and compute holding value ----------
dfw['_manager_key'] = dfw[mgr_col].astype(str).map(norm_name)

if val_col is not None:
    dfw['holding_value'] = pd.to_numeric(dfw[val_col], errors='coerce')
else:
    shares = pd.to_numeric(dfw[sh_col], errors='coerce')
    price  = pd.to_numeric(dfw[pr_col], errors='coerce')
    dfw['holding_value'] = shares * price

dfw['holding_value'] = dfw['holding_value'].clip(lower=0).fillna(0)

# ---------- aggregate to AUM per manager ----------
if dt_col is not None:
    # per (manager, date) sum; then take median across dates for stability
    g = (dfw.groupby(['_manager_key', dfw[dt_col].astype(str)])['holding_value']
             .sum()
             .groupby(level=0)
             .median())
    manager_aum = g
else:
    # single snapshot
    manager_aum = dfw.groupby('_manager_key')['holding_value'].sum()

# scale sanity (optional): convert to float
manager_aum = manager_aum.astype(float)

# ---------- inject into centrality DF ----------
C['market_value'] = manager_aum.reindex(C.index).astype(float)
cov = C['market_value'].notna().sum()
print(f"[AUM] Coverage: {cov} / {len(C)} managers ({cov/len(C):.1%})")

# ---------- correlations: raw + log1p ----------
def corr_stats(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y) & (y > 0)
    if mask.sum() < 10:
        return np.nan, np.nan, np.nan, np.nan, int(mask.sum())
    pear = np.corrcoef(x[mask], y[mask])[0, 1]
    spear = spearmanr(x[mask], y[mask]).correlation
    lx, ly = np.log1p(x[mask]), np.log1p(y[mask])
    lpear = np.corrcoef(lx, ly)[0, 1]
    lspear = spearmanr(lx, ly).correlation
    return pear, spear, lpear, lspear, int(mask.sum())

rows = []
for metric in [c for c in C.columns if c != 'market_value']:
    pear, spear, lpear, lspear, n = corr_stats(C[metric].values, C['market_value'].values)
    rows.append({
        'metric': metric,
        'pearson': pear, 'spearman': spear,
        'pearson_log1p': lpear, 'spearman_log1p': lspear,
        'n_pairs': n
    })

plaus_table = (pd.DataFrame(rows)
                 .sort_values(['spearman_log1p','spearman'], ascending=False))

print("\n=== (c) Plausibility vs AUM (higher = better; log1p is more robust) ===")
print(plaus_table.to_string(index=False, float_format=lambda z: f"{z:,.4f}"))


[AUM] Using WRDS file: ../../WRDS/final_deduplicated_holdings.csv
[AUM] Coverage: 1087 / 5717 managers (19.0%)

=== (c) Plausibility vs AUM (higher = better; log1p is more robust) ===
        metric  pearson  spearman  pearson_log1p  spearman_log1p  n_pairs
   eigenvector   0.1700    0.6211         0.1130          0.6211     1087
   w_in_degree   0.6089    0.6148         0.6056          0.6148     1087
hits_authority   0.7449    0.6004         0.2656          0.6004     1087
      pagerank   0.6659    0.5674         0.3443          0.5674     1087
     in_degree   0.6457    0.4915         0.5722          0.4915     1087
  w_out_degree   0.0574    0.1108         0.1216          0.1108     1087
    out_degree   0.0595    0.1108         0.1316          0.1108     1087
      hits_hub   0.0473    0.1108         0.1029          0.1108     1087
   betweenness   0.0465    0.1045         0.0785          0.1045     1087
